========================================================================================
Get the residual only from test set and save them for ML 
========================================================================================
Save the residuals separately for training and test set for PCA training and test
========================================================================================

In [ ]:
import sys
sys.path.append('working_path')

import numpy as np
import pandas as pd
import os

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.multioutput import MultiOutputClassifier, MultiOutputRegressor
from sklearn.compose import ColumnTransformer

In [ ]:
df = pd.read_csv('working_path/cross_reversed.csv')

In [ ]:
"""
Define variables:
X set is sleep health
y set is depression
Age and sex are confounds 
"""
# Define variables
Age = ['age']
Sex = ['Sex']
sleep = ['sleep_variables']
N_12 = ['n_12_depression_variables']
RDS_4 = ['rds4_depression_variables']
depression = N_12 + RDS_4

X_set = sleep
y_set = depression
confounds = Age # Sex not included for normalization 

# Extract numerical data for X_set, y_set, and confounds
X_data = df[X_set].values
y_data = df[y_set].values

In [5]:
# Define feature group for scaling
features_to_scale = X_set + confounds
features_to_skip = Sex

# Setup the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), features_to_scale)
    ],
    remainder='passthrough' # Sex is binary
)

In [ ]:
"""
To keep the same structure of the data in sleep-related depression, the dataset will be splitted to 5 folds, each of them will be applied for multivariate GLM and the other 4 for ML.
Residuals from multivaraite GLM will be the target for later ML.
"""
# Define cross-validation strageties
# Define the CV for the entire dataset
n_splits_entire = 5

random_seed = 42

Outer_cv_entire = KFold(n_splits=n_splits_entire, shuffle=True, random_state=random_seed)

# Define grid of potential regularization parameters for logistic and regid regression
param_grid = {'estimator__C': [0.001, 0.01, 0.1, 1, 10]}
alpha_grid = {'estimator__alpha': np.logspace(-4, 0, 10)}

In [ ]:
'''
The residuals will be calculated and saved separately for training and test set for later PCA
'''
# Train multivariateGLM in inner CV and get the residuals for the test sets in outer CV
for i, (ML_idx, GLM_idx) in enumerate(Outer_cv_entire.split(df)):
    
    # Split data 
    df_GLM_train = df.iloc[GLM_idx]
    df_ML_test = df.iloc[ML_idx]

    # Scaling
    X_train_raw = df_GLM_train[features_to_scale + features_to_skip]
    X_test_raw = df_ML_test[features_to_scale + features_to_skip]

    X_train_s = preprocessor.fit_transform(X_train_raw)
    X_test_s = preprocessor.transform(X_test_raw) 

    # Prepare Targets
    Y_train = y_data[GLM_idx]
    Y_test = y_data[ML_idx]
    
    Y_bin_train, Y_ord_train = Y_train[:, :len(N_12)], Y_train[:, len(N_12):]
    Y_bin_test, Y_ord_test = Y_test[:, :len(N_12)], Y_test[:, len(N_12):]

    '''
    Binary target 
    '''
    log_reg = MultiOutputClassifier(LogisticRegression(penalty='l2'))
    grid_bin = GridSearchCV(log_reg, param_grid=param_grid, scoring='accuracy')
    grid_bin.fit(X_train_s, Y_bin_train)

    # Calculate residuals on the training set
    Y_bin_train_prob = [est.predict_proba(X_train_s)[:, 1] for est in grid_bin.best_estimator_.estimators_]
    Y_bin_train_prob_array = np.array(Y_bin_train_prob).T
    res_binary_train = Y_bin_train - Y_bin_train_prob_array

    # Predict probabilities on the test set
    Y_bin_test_prob = [est.predict_proba(X_test_s)[:, 1] for est in grid_bin.best_estimator_.estimators_]
    Y_bin_test_prob_array = np.array(Y_bin_test_prob).T
    res_binary_test = Y_bin_test - Y_bin_test_prob_array

    '''
    Ordinal target
    '''
    ridge = MultiOutputRegressor(Ridge())
    grid_ord = GridSearchCV(ridge, param_grid=alpha_grid, scoring='neg_mean_squared_error')
    grid_ord.fit(X_train_s, Y_ord_train)

    # Calculate residuals on the traning set
    Y_ord_train_pred = grid_ord.predict(X_train_s)
    res_ordinal_train = Y_ord_train - Y_ord_train_pred

    # Predict on the test set
    Y_ord_test_pred = grid_ord.predict(X_test_s)
    res_ordinal_test = Y_ord_test - Y_ord_test_pred

    # Print out residuals
    print(f'Projected residuals for depression for fold {i+1}: {res_ordinal_test}')

    '''
    Save Projected Residuals
    '''
    # Combine all residuals
    all_res_test = np.column_stack((res_binary_test, res_ordinal_test))
    all_res_train = np.column_stack((res_binary_train, res_ordinal_train))

    # Assign variable names for residuals
    res_cols = [f"{col}_res" for col in (N_12 + RDS_4)]

    # Assign the residuals to the main df for the matching rows
    df_GLM = df.iloc[GLM_idx].copy()
    df_ML = df.iloc[ML_idx].copy()

    df_GLM[res_cols] = all_res_train
    df_ML[res_cols] = all_res_test

    # Save to CSV
    filename_GLM = f'save_path/depression_residuals_training_fold_{i+1}.csv'
    filename_ML = f'save_path/depression_residuals_test_fold_{i+1}.csv'
    df_GLM.to_csv(filename_GLM, index=False)
    df_ML.to_csv(filename_ML, index=False)